### HuggingFaceEmbeddings

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_teddynote import logging
import os
import warnings

logging.langsmith("CH08-Embeddings")

warnings.filterwarnings("ignore") # 경고 무시

os.environ["HF_HOME"] = "./cache/" # 다운로드 경로 설정

LangSmith 추적을 시작합니다.
[프로젝트명]
CH08-Embeddings


In [3]:
texts = [
    "안녕, 만나서 반가워.",
    "LangChain simplifies the process of building applications with large language models.",
    "랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다.",
    "LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.",
    "Retrieval-Augmented Generation (RAG) is an effective technique for imporving AI responses.",
]

In [5]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings

model_name = "intfloat/multilingual-e5-large-instruct"

hf_embedding = HuggingFaceEndpointEmbeddings(
    model=model_name,
    task="feature-extraction",
    huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"],
)

In [9]:
%%time
embedded_documents = hf_embedding.embed_documents(texts) # 문서 임베딩 생성


CPU times: total: 188 ms
Wall time: 1.43 s


HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/hf-inference/models/intfloat/multilingual-e5-large-instruct/pipeline/feature-extraction' (Request ID: Root=1-6ab1ea35-3ca7319b2203721a17e0c68d;ac564d07-0d3b-4264-8352-bb6c6edea417)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

### UpstageEmbeddings
- llm과 문서 ai 분야에 특화된 국내 스타트업

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")


In [4]:
texts = [
    "안녕, 만나서 반가워.",
    "LangChain simplifies the process of building applications with large language models.",
    "랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다.",
    "LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.",
    "Retrieval-Augmented Generation (RAG) is an effective technique for imporving AI responses.",
]

In [5]:
from langchain_upstage import UpstageEmbeddings
from langchain_teddynote import logging
from dotenv import load_dotenv

logging.langsmith("CH08-Embeddings")
load_dotenv()

#쿼리 전용 임베딩 모델
query_embeddings = UpstageEmbeddings(model="solar-embedding-1-large-query")

# 문서 전용 임베딩 모델
passage_embeddings = UpstageEmbeddings(model="solar-embedding-1-large-passage")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH08-Embeddings


In [6]:
embedded_query = query_embeddings.embed_query("LangChain에 대해서 상세히 알려주세요.")
len(embedded_query)

4096

In [11]:
embedded_documents = passage_embeddings.embed_documents(texts) # 문서 임베딩 생성

In [12]:
import numpy as np

# 질문(embedded_query): LangChain에 대해서 알려주세요.
similarity = np.array(embedded_query) @ np.array(embedded_documents).T

# 유사도 기준 내림차순 정렬
sorted_idx = (np.array(embedded_query) @ np.array(embedded_documents).T).argsort()[::-1]

# 결과 출력
print("[Query] Langchain에 대해서 알려주세요.\n===============================")
for i, idx in enumerate(sorted_idx):
    print(f"[{i}] 유사도: {similarity[idx]:.3f}, 문서: {texts[idx]}")

[Query] Langchain에 대해서 알려주세요.
[0] 유사도: 0.484, 문서: LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.
[1] 유사도: 0.476, 문서: LangChain simplifies the process of building applications with large language models.
[2] 유사도: 0.433, 문서: 랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다.
[3] 유사도: 0.183, 문서: Retrieval-Augmented Generation (RAG) is an effective technique for imporving AI responses.
[4] 유사도: 0.153, 문서: 안녕, 만나서 반가워.
